In [1]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       4.2Gi       5.0Gi        62Mi       3.5Gi       8.1Gi
Swap:             0B          0B          0B


In [2]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

wheels_dir = "/content/drive/MyDrive/Colab Notebooks/wheels/"
!mkdir -p "{wheels_dir}"

Mounted at /content/drive


In [6]:
packages = [
    "pandas", "faiss-cpu",
    "sentence-transformers",
    "python-docx"
]

!pip download {" ".join(packages)} -d "{wheels_dir}"
!pip install --no-index --find-links="{wheels_dir}" {' '.join(packages)}

  File was already downloaded /content/drive/MyDrive/Colab Notebooks/wheels/pandas-3.0.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl
  File was already downloaded /content/drive/MyDrive/Colab Notebooks/wheels/faiss_cpu-1.14.3-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
  File was already downloaded /content/drive/MyDrive/Colab Notebooks/wheels/sentence_transformers-5.5.1-py3-none-any.whl
  File was already downloaded /content/drive/MyDrive/Colab Notebooks/wheels/python_docx-1.2.0-py3-none-any.whl
  File was already downloaded /content/drive/MyDrive/Colab Notebooks/wheels/numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
  File was already downloaded /content/drive/MyDrive/Colab Notebooks/wheels/python_dateutil-2.9.0.post0-py2.py3-none-any.whl
  File was already downloaded /content/drive/MyDrive/Colab Notebooks/wheels/packaging-26.2-py3-none-any.whl
  File was already downloaded /content/drive/MyDrive/Colab Notebooks/wheels/transforme

In [7]:
# !pip wheel llama-cpp-python -w "{wheels_dir}"

# !find /content -name "*llama_cpp_python*.whl" 2>/dev/null

# !mv /content/{wheel_dir}/* "{wheels_dir}"

!pip install "{wheels_dir}"llama_cpp_python-0.3.28-py3-none-linux_x86_64.whl

Processing ./drive/MyDrive/Colab Notebooks/wheels/llama_cpp_python-0.3.28-py3-none-linux_x86_64.whl
llama-cpp-python is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.


In [8]:
import os
HF_CACHE_DIR = "/content/drive/MyDrive/Colab Notebooks/hf_models/"

os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = f"{HF_CACHE_DIR}/transformers/"
os.environ["HF_HUB_CACHE"] = f"{HF_CACHE_DIR}/hub/"

from sentence_transformers import SentenceTransformer

In [9]:
embedding_model_id = "BAAI/bge-base-en-v1.5"

def save_embedding_model(model_id, save_dir):

    embedding_model = SentenceTransformer(model_id, device="cpu")

    model_name = model_id.split("/")[-1]
    save_dir += model_name

    embedding_model.save(save_dir)

    print(f"Saved to: {save_dir}")

def load_embedding_model(model_id, model_dir):

    model_name = model_id.split("/")[-1]

    model_dir += model_name
    embedding_model = SentenceTransformer(model_dir, device="cpu")

    return embedding_model

def generate_embeddings(embedding_model, texts):

    embeddings = embedding_model.encode(
        texts,
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    return embeddings

# save_embedding_model(model_id=embedding_model_id, save_dir=HF_CACHE_DIR)
embedding_model = load_embedding_model(model_id=embedding_model_id, model_dir=HF_CACHE_DIR)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [40]:
job_descriptions = [
    "Senior Backend Engineer with Python, AWS and Docker",
    "Data Engineer with Spark and Databricks",
    "ML Engineer with LLM and RAG experience"
]

emb_vectors = generate_embeddings(embedding_model=embedding_model, texts=job_descriptions)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [10]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from torch import no_grad, sigmoid, tensor

reranking_model_id = "BAAI/bge-reranker-v2-m3"

def save_hf_transformer_model(model_id, save_dir):

    model = AutoModelForSequenceClassification.from_pretrained(model_id)
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    model_name = model_id.split("/")[-1]
    save_dir += model_name

    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)

    print(f"Saved to: {save_dir}")

def load_hf_transformer_model(model_id, model_dir):

    model_name = model_id.split("/")[-1]

    tokenizer = AutoTokenizer.from_pretrained(f"{model_dir}{model_name}")
    model = AutoModelForSequenceClassification.from_pretrained(f"{model_dir}{model_name}")

    model.eval()

    return tokenizer, model

def rerank_documents(reranker_model, reranker_tokenizer, query, documents, batch_size=16):

    relevance_scores = []
    pairs = [[query, document] for document in documents]

    with no_grad():

        for start in range(0, len(documents), batch_size):

            batch = [[query, document] for document in documents[start:start+batch_size]]

            inputs = reranker_tokenizer(
                batch,
                padding=True, truncation=True,
                max_length=512, return_tensors="pt"
            )

            outputs = reranker_model(**inputs)

            relevance_scores += outputs.logits.view(-1).float().tolist()

    relevance_scores = sigmoid(tensor(relevance_scores)).tolist()

    return relevance_scores

# save_hf_transformer_model(model_id=reranking_model_id, save_dir=HF_CACHE_DIR)
reranker_tokenizer, reranker_model = load_hf_transformer_model(model_id=reranking_model_id, model_dir=HF_CACHE_DIR)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [30]:
query = """
Software engineer with 5 years experience building RAG systems,
LangChain applications, vector databases, Python backend services
and LLM-based products.
"""

documents = [
    "Senior Python Backend Engineer. Skills: Python, FastAPI, PostgreSQL, AWS.",
    "Data Engineer. Skills: Spark, Databricks, Airflow, Python.",
    "LLM Engineer. Skills: RAG, LangGraph, Vector Databases, Python.",
    "Frontend Engineer. Skills: React, TypeScript, CSS.",
    "DevOps Engineer. Skills: Kubernetes, Docker, Terraform, AWS."
]

reranked_scores = rerank_documents(reranker_model, reranker_tokenizer, query, documents, batch_size=16)

In [19]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

llm_model_id = "Qwen/Qwen3-4B-GGUF"
model_name = llm_model_id.split("/")[-1]
model_file = "Qwen3-4B-Q4_K_M.gguf"

if not os.path.exists(f"/content/{model_file}"):
    !cp "{HF_CACHE_DIR + model_name}"/"{model_file}" /content/

def save_llm_model(model_id, model_file, save_dir):

    model_name = model_id.split("/")[-1]
    save_dir += model_name

    hf_hub_download(
        repo_id=model_id,
        filename=model_file,
        local_dir=save_dir,
        local_dir_use_symlinks=False
    )

    print(f"Saved to: {save_dir}")

def load_llm_model(model_file, context_window, verbose=False):

    model_file = "/content/" + model_file

    llm_model = Llama(
        model_path=model_file,
        n_ctx=context_window,
        n_threads=os.cpu_count(),
        verbose=verbose
    )

    return llm_model

def run_llm_inference(llm_model, system_prompt, user_prompt, max_tokens=1024, temperature=0.3):

    response = llm_model.create_chat_completion(
        messages=[
            {"role": "system", "content": "\n".join([system_prompt, "/no_think"])},
            {"role": "user", "content": "\n".join([user_prompt, "/no_think"])}
        ],
        max_tokens=max_tokens, temperature=temperature
    )

    print(response)

    response_text = response["choices"][0]["message"]["content"]

    return response_text, response

# save_llm_model(model_id=llm_model_id, model_file=model_file, save_dir=HF_CACHE_DIR)
llm_model = load_llm_model(model_file=model_file, context_window=8192, verbose=False)

llama_context: n_ctx_seq (8192) < n_ctx_train (40960) -- the full capacity of the model will not be utilized


In [86]:
response = run_llm_inference(
    llm_model=llm_model,
    system_prompt="You are a technical writer",
    user_prompt="Write a markdown overview of vector databases.",
    max_tokens=1024, temperature=0.3
)

print(response)

{'id': 'chatcmpl-0aaf6d5a-e21b-48e1-9ce8-27d28c3b2cf7', 'object': 'chat.completion', 'created': 1781386331, 'model': '/content/Qwen3-4B-Q4_K_M.gguf', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '<think>\n\n</think>\n\n# Overview of Vector Databases\n\n## What is a Vector Database?\n\nA **vector database** is a type of database optimized for storing and retrieving **vector data** — numerical arrays that represent high-dimensional data, such as images, text, audio, and other unstructured or semi-structured data. These databases are designed to efficiently handle **vector similarity searches**, which are essential for applications like **retrieval augmented generation (RAG)**, **recommendation systems**, **image search**, and **natural language processing (NLP)**.\n\n## Key Features of Vector Databases\n\n| Feature | Description |\n|--------|-------------|\n| **Vector Storage** | Stores high-dimensional vectors (e.g., 768-dimensional embeddings) in a structured wa

In [12]:
from docx import Document
from docx.document import Document as DocxDocument
from docx.table import Table
from docx.text.paragraph import Paragraph

def iter_block_items(parent):
    for child in parent.element.body:
        if child.tag.endswith("p"):
            yield Paragraph(child, parent)
        elif child.tag.endswith("tbl"):
            yield Table(child, parent)

def extract_text_from_docx(file_path):

    document = Document(file_path)

    contents = []

    for block in iter_block_items(document):

        if isinstance(block, Paragraph):
            if block.text.strip():
                contents.append(block.text)

        elif isinstance(block, Table):
            for row in block.rows:
                row_text = " | ".join(
                    cell.text.strip()
                    for cell in row.cells
                )
                contents.append(row_text)

    return "\n".join(contents)

contents = extract_text_from_docx("job_description.docx")

In [20]:
system_prompt = """
You are an expert Human Resources Assistant who is expertised in identifying every detail about a job requirement.
Given a job decription, you will identify every single requirement from it.
When you identify the job requirements, you must clearly recognize both wanted and unwanted factors that are expected in a candidate.
Group the identified requirements based on most relevant topics.

You must always produce the complete requirements list strictly in markdown text format with the following specifications:
* You will create two lists one list for wanted factors and another for unwanted factors with both list names having H2 formatting (## ).
* Each of the two lists will have the relevant topic of requirements in H3 formatting (### ).
* Each requirement topic shall be having a bulleted list of requirements relevant to that topic under each of the two lists.
* You must list only one requirement per bullet point and never combine multiple requirements into a single bullet point under any topic.
"""

user_prompt = f"""
Here are the details from a job description:
{contents}

Extract all the job requirements by grouping them into relevant topics under each of the wanted and unwanted factors lists in the expected markdown format.
"""

messages=[
    {"role": "system", "content": "\n".join([system_prompt, "/no_think"])},
    {"role": "user", "content": "\n".join([user_prompt, "/no_think"])}
]

from json import dumps
len(llm_model.tokenize(dumps(messages).encode("utf-8")))

2501

In [21]:
from time import time

start = time()

response, response_with_meta = run_llm_inference(
    llm_model=llm_model,
    system_prompt=system_prompt,
    user_prompt=user_prompt,
    max_tokens=4096, temperature=0.3
)

response_time_secs = time() - start
tokens_rate = response_with_meta["usage"]["total_tokens"] / response_time_secs

print(response, '\n\n', tokens_rate)

{'id': 'chatcmpl-3e1bab7d-70c3-475c-aa5f-d46e5da8edd4', 'object': 'chat.completion', 'created': 1781390085, 'model': '/content/Qwen3-4B-Q4_K_M.gguf', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '<think>\n\n</think>\n\n## Wanted Factors\n\n### Technical Experience\n* Production experience with embeddings-based retrieval systems (sentence-transformers, OpenAI embeddings, BGE, E5, or similar) deployed to real users.\n* Production experience with vector databases or hybrid search infrastructure (Pinecone, Weaviate, Qdrant, Milvus, OpenSearch, Elasticsearch, FAISS, or something similar).\n* Strong Python.\n\n### System Design and Evaluation\n* Hands-on experience designing evaluation frameworks for ranking systems (NDCG, MRR, MAP, offline-to-online correlation, A/B test interpretation).\n* Experience with learning-to-rank models (XGBoost-based or neural).\n\n### Product and Engineering Attitude\n* Scrappy product-engineering attitude — willing to ship a working rank

In [16]:
print(response)

<think>

</think>

## Wanted Factors

### Technical Experience
* Production experience with embeddings-based retrieval systems (sentence-transformers, OpenAI embeddings, BGE, E5, or similar) deployed to real users.
* Production experience with vector databases or hybrid search infrastructure (Pinecone, Weaviate, Qdrant, Milvus, OpenSearch, Elasticsearch, FAISS, or something similar).
* Strong Python.

### System Design and Evaluation
* Hands-on experience designing evaluation frameworks for ranking systems (NDCG, MRR, MAP, offline-to-online correlation, A/B test interpretation).
* Experience with learning-to-rank models (XGBoost-based or neural).

### Product and Engineering Attitude
* Scrappy product-engineering attitude — willing to ship a working ranker in a week even if the underlying ML is "obviously suboptimal," because we need to learn from real users before we know what to actually optimize for.

### Long-Term Vision and Leadership
* Driving the long-term architecture of how we

In [18]:
response_time_secs

895.6869966983795